<a href="https://colab.research.google.com/github/christinChris/gaiascope/blob/main/notebooks/BurnScope_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import ee, geemap
from google.colab import auth

try:
    auth.authenticate_user()
    ee.Initialize(project="ee-christilekka")
    print("Connected to Earth Engine")
except Exception as e:
    print("Connection error:", e)

m = geemap.Map(center=[38.22, 23.93], zoom=12)
m.add_basemap("HYBRID")
m

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Connected to Earth Engine


Map(center=[38.22, 23.93], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [4]:
aoi = ee.Geometry.Rectangle([23.80, 38.05, 24.05, 38.30])

before = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(aoi)
          .filterDate("2024-07-20", "2024-08-10")
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
          .median()
          .clip(aoi))

m.centerObject(aoi, 11)
m.addLayer(before, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "Before · true color")
m

Map(center=[38.17499478165861, 23.92499999999964], controls=(WidgetControl(options=['position', 'transparent_b…

In [5]:
after = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(aoi)
         .filterDate("2024-08-15", "2024-09-05")
         .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
         .median()
         .clip(aoi))

m.addLayer(after, {"bands": ["B12", "B8", "B4"], "min": 0, "max": 3000}, "After · false color")
m

Map(center=[38.17499478165861, 23.92499999999964], controls=(WidgetControl(options=['position', 'transparent_b…

In [6]:
def s2_composite(start, end, area):
    return (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(area)
            .filterDate(start, end)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
            .median()
            .clip(area))

aoi = ee.Geometry.Rectangle([23.75, 37.98, 24.10, 38.32])

before = s2_composite("2024-07-20", "2024-08-10", aoi)
after  = s2_composite("2024-08-15", "2024-09-05", aoi)

m = geemap.Map()
m.add_basemap("HYBRID")
m.centerObject(aoi, 11)
m.addLayer(before, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "Before · true color")
m.addLayer(after,  {"bands": ["B12", "B8", "B4"], "min": 0, "max": 3000}, "After · false color")
m

Map(center=[38.14999775240434, 23.924999999999493], controls=(WidgetControl(options=['position', 'transparent_…

In [7]:
def nbr(img):
    return img.normalizedDifference(["B8", "B12"]).rename("NBR")

dnbr  = nbr(before).subtract(nbr(after)).rename("dNBR")
burnt = dnbr.gt(0.27)

m.addLayer(dnbr, {"min": -0.1, "max": 0.8,
                  "palette": ["white", "yellow", "orange", "red", "black"]}, "dNBR")
m.addLayer(burnt.selfMask(), {"palette": ["#C2410C"]}, "Burnt (dNBR > 0.27)")
m

Map(center=[38.14999775240434, 23.924999999999493], controls=(WidgetControl(options=['position', 'transparent_…

In [8]:
lc   = ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi)
land = lc.neq(80)

burnt = dnbr.gt(0.27).And(land)

m.addLayer(burnt.selfMask(), {"palette": ["#C2410C"]}, "Burnt · land only")
m

Map(center=[38.14999775240434, 23.924999999999493], controls=(WidgetControl(options=['position', 'transparent_…

In [9]:
burnt_m2 = (burnt.rename("burnt")
            .multiply(ee.Image.pixelArea())
            .reduceRegion(reducer=ee.Reducer.sum(),
                          geometry=aoi, scale=20, maxPixels=1e10)
            .get("burnt"))

km2 = ee.Number(burnt_m2).divide(1e6).getInfo()
print(f"Burnt area: {km2:.1f} km²")

Burnt area: 95.4 km²


In [10]:
import pandas as pd

classes = {10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
           50: "Built-up", 60: "Bare / sparse", 70: "Snow and ice",
           80: "Permanent water", 90: "Herbaceous wetland", 95: "Mangroves",
           100: "Moss and lichen"}

groups = (ee.Image.pixelArea().updateMask(burnt)
          .addBands(lc)
          .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
                        geometry=aoi, scale=20, maxPixels=1e10)
          .get("groups")
          .getInfo())

rows = [{"Land cover": classes[g["class"]], "Area (km²)": g["sum"] / 1e6} for g in groups]

df = pd.DataFrame(rows)
df = df[df["Area (km²)"] >= 0.05]
df = df.sort_values("Area (km²)", ascending=False)
df["Share (%)"] = df["Area (km²)"] / df["Area (km²)"].sum() * 100
df.loc[len(df)] = ["Total", df["Area (km²)"].sum(), 100.0]

df.style.format(precision=1).hide(axis="index")

Land cover,Area (km²),Share (%)
Shrubland,48.1,50.5
Tree cover,35.5,37.2
Grassland,10.5,11.0
Cropland,0.6,0.7
Built-up,0.6,0.6
Total,95.3,100.0


In [11]:
SEVERITY   = {1: "Low", 2: "Moderate-low", 3: "Moderate-high", 4: "High"}
SEV_COLORS = ["#FFD54F", "#FF8F00", "#D84315", "#7F0000"]

def lc_table(groups):
    rows = [{"Land cover": classes[g["class"]], "Area (km²)": g["sum"] / 1e6} for g in groups]
    df = pd.DataFrame(rows)
    df = df[df["Area (km²)"] >= 0.05]
    df = df.sort_values("Area (km²)", ascending=False)
    df["Share (%)"] = df["Area (km²)"] / df["Area (km²)"].sum() * 100
    df.loc[len(df)] = ["Total", df["Area (km²)"].sum(), 100.0]
    return df.style.format(precision=1).hide(axis="index")


def severity_df(groups):
    rows = [{"Land cover": classes[int(g["combo"]) // 10],
             "Severity":   SEVERITY[int(g["combo"]) % 10],
             "km²":        g["sum"] / 1e6} for g in groups]
    df = pd.DataFrame(rows).pivot_table(index="Land cover", columns="Severity",
                                        values="km²", aggfunc="sum", fill_value=0)
    df = df[[s for s in SEVERITY.values() if s in df.columns]]
    df = df[df.sum(axis=1) >= 0.05]
    return df.loc[df.sum(axis=1).sort_values().index]


def burnscope(aoi, before_dates, after_dates, threshold=0.27):
    # 1. Images
    before = s2_composite(before_dates[0], before_dates[1], aoi)
    after  = s2_composite(after_dates[0],  after_dates[1],  aoi)

    # 2. Science: dNBR, burnt mask (land only), severity classes
    dnbr  = nbr(before).subtract(nbr(after)).rename("dNBR")
    lc    = ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi)
    burnt = dnbr.gt(threshold).And(lc.neq(80)).rename("burnt")
    sev   = (ee.Image(1).where(dnbr.gte(0.27), 2)
                        .where(dnbr.gte(0.44), 3)
                        .where(dnbr.gte(0.66), 4)
                        .updateMask(burnt).rename("severity"))

    # 3. Numbers: total, per land cover, per land cover × severity
    area  = ee.Image.pixelArea().updateMask(burnt)
    total = area.reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi,
                              scale=20, maxPixels=1e10).get("area")
    groups = (area.addBands(lc)
              .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="class"),
                            geometry=aoi, scale=20, maxPixels=1e10)
              .get("groups"))
    combo = lc.toInt().multiply(10).add(sev).rename("combo")
    sev_groups = (area.addBands(combo)
                  .reduceRegion(reducer=ee.Reducer.sum().group(groupField=1, groupName="combo"),
                                geometry=aoi, scale=20, maxPixels=1e10)
                  .get("groups"))

    # 4. One trip to Google's servers
    info = ee.Dictionary({"total": total, "groups": groups, "sev_groups": sev_groups}).getInfo()

    return {"before": before, "after": after, "dnbr": dnbr, "burnt": burnt, "severity": sev,
            "km2": info["total"] / 1e6, "table": lc_table(info["groups"]),
            "sev_df": severity_df(info["sev_groups"])}

In [12]:
r = burnscope(aoi, ("2024-07-20", "2024-08-10"), ("2024-08-15", "2024-09-05"), threshold=0.44)

print(f"Burnt area: {r['km2']:.1f} km²")
r["table"]

Burnt area: 71.5 km²


Land cover,Area (km²),Share (%)
Shrubland,35.6,49.9
Tree cover,31.4,44.0
Grassland,4.0,5.7
Cropland,0.2,0.3
Built-up,0.2,0.3
Total,71.5,100.0


In [13]:
!pip install -q solara

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.4/138.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.5/277.5 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.5 MB/s eta 0:00:00


In [14]:
from google.colab import output
output.enable_custom_widget_manager()

import solara, ipyleaflet
threshold = solara.reactive(0.27)

In [15]:
from google.colab import output
output.enable_custom_widget_manager()

In [25]:
import solara, ipyleaflet
import matplotlib.pyplot as plt

# ---------- Event catalog ----------
FIRE_EVENTS = {
    # ---------- 2021 ----------
    "Varympompi · Athens (Aug 2021)": {
        "before": ("2021-07-15", "2021-08-01"),
        "after":  ("2021-08-10", "2021-08-31"),
        "bbox":   [23.70, 38.10, 23.85, 38.20],
        "info":   "One of the extreme wildfires of early August 2021, it burned through the northern suburbs "
                  "of Athens around Varympompi and Tatoi.",
        "source": "https://www.mdpi.com/2673-4931/17/1/8",
    },
    "North Evia · Greece (Aug 2021)": {
        "before": ("2021-07-15", "2021-08-02"),
        "after":  ("2021-08-20", "2021-09-10"),
        "bbox":   [23.05, 38.80, 23.45, 39.05],
        "info":   "In early August 2021, during the Mediterranean heatwave, huge wildfires burned large parts "
                  "of northern Evia's pine forests and villages.",
        "source": "https://www.esa.int/ESA_Multimedia/Images/2021/08/Wildfires_ravage_Greek_island_of_Evia",
    },

    # ---------- 2023 ----------
    "Dervenochoria · Attica (Jul 2023)": {
        "before": ("2023-07-01", "2023-07-15"),
        "after":  ("2023-07-25", "2023-08-15"),
        "bbox":   [23.30, 38.08, 23.60, 38.25],
        "info":   "One of several fires that started in Attica on 17 July 2023, during a strong heatwave, "
                  "burning around the villages of Dervenochoria in western Attica.",
        "source": "https://mapping.emergency.copernicus.eu/news/information-bulletin-169-the-copernicus-emergency-management-service-maps-some-critical-wildfires-in-greece-update/",
    },
    "Rhodes · Greece (Jul 2023)": {
        "before": ("2023-07-01", "2023-07-15"),
        "after":  ("2023-08-01", "2023-08-20"),
        "bbox":   [27.80, 35.90, 28.25, 36.35],
        "info":   "A wildfire started on 18 July 2023 in central Rhodes and spread south over the following days, "
                  "forcing thousands of residents and tourists to evacuate.",
        "source": "https://mapping.emergency.copernicus.eu/news/wildfire-on-rhodes-island-greece/",
    },
    "Corfu · Ionian Islands (Jul 2023)": {
        "before": ("2023-07-01", "2023-07-21"),
        "after":  ("2023-07-28", "2023-08-20"),
        "bbox":   [19.80, 39.65, 19.95, 39.80],
        "info":   "Wildfires in north-eastern Corfu led to mass evacuations on 23 July 2023, "
                  "with residents directed towards Ipsos and Barbati.",
        "source": "https://mapping.emergency.copernicus.eu/news/information-bulletin-169-the-copernicus-emergency-management-service-maps-some-critical-wildfires-in-greece-update/",
    },
    "Magnesia · Thessaly (Jul 2023)": {
        "before": ("2023-07-05", "2023-07-24"),
        "after":  ("2023-08-01", "2023-08-25"),
        "bbox":   [22.65, 39.15, 22.95, 39.40],
        "info":   "Fires that began on 26 July 2023 near Almyros and Velestino reached the Nea Anchialos "
                  "air base and forced the evacuation of the town.",
        "source": "https://en.wikipedia.org/wiki/2023_Greece_wildfires",
    },
    "Evros – Dadia · Greece (Aug 2023)": {
        "before": ("2023-07-20", "2023-08-15"),
        "after":  ("2023-09-05", "2023-09-30"),
        "bbox":   [25.55, 40.80, 26.50, 41.35],
        "info":   "The Evros fire began on 21 August 2023 and burned for weeks across the Dadia forest "
                  "and the area north of Alexandroupolis. Around 73,000 hectares burned; EU officials called it "
                  "the largest wildfire recorded in the EU.",
        "source": "https://en.wikipedia.org/wiki/2023_Greece_wildfires",
    },
    "Parnitha – Fyli · Attica (Aug 2023)": {
        "before": ("2023-08-01", "2023-08-20"),
        "after":  ("2023-08-26", "2023-09-15"),
        "bbox":   [23.60, 38.08, 23.80, 38.22],
        "info":   "A wildfire started on 22 August 2023 near a monastery in Fyli and reached "
                  "Mount Parnitha the next day, burning part of the national park before it was put out on 24 August.",
        "source": "https://en.wikipedia.org/wiki/2023_Greece_wildfires",
    },

    # ---------- 2024 ----------
    "Varnavas · Attica (Aug 2024)": {
        "before": ("2024-07-20", "2024-08-10"),
        "after":  ("2024-08-15", "2024-09-05"),
        "bbox":   [23.75, 37.98, 24.10, 38.32],
        "info":   "A wildfire broke out near Varnavas, north-east of Athens, on 11 August 2024 "
                  "and spread rapidly south towards the northern suburbs of Athens.",
        "source": "https://mapping.emergency.copernicus.eu/news/wildfire-in-attica-region-greece/",
    },
    "Corinthia · Peloponnese (Sep 2024)": {
        "before": ("2024-09-01", "2024-09-27"),
        "after":  ("2024-10-05", "2024-10-25"),
        "bbox":   [22.45, 37.90, 22.75, 38.12],
        "info":   "A wildfire started near the village of Rozena in Corinthia on 29 September 2024 "
                  "and spread across the northern Peloponnese.",
        "source": "https://reliefweb.int/report/greece/greece-wildfire-update-greek-civil-protection-copernicus-emsr-greek-government-jrc-effis-media-echo-daily-flash-03-october-2024",
    },
}

# ---------- State ----------
mode         = solara.reactive("Map your own")
event        = solara.reactive(list(FIRE_EVENTS)[-1])
before_start = solara.reactive("2024-07-20")
before_end   = solara.reactive("2024-08-10")
after_start  = solara.reactive("2024-08-15")
after_end    = solara.reactive("2024-09-05")
threshold  = solara.reactive(0.27)
basemap    = solara.reactive("Google satellite")
show_burnt = solara.reactive(True)
area       = solara.reactive(ee.Geometry.Rectangle([23.75, 37.98, 24.10, 38.32]))
result     = solara.reactive(None)
status     = solara.reactive("")

BASEMAPS = {
    "Google satellite": "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
    "Google map":       "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
    "Google terrain":   "https://mt1.google.com/vt/lyrs=p&x={x}&y={y}&z={z}",
}
TRUE_COLOR  = {"bands": ["B4", "B3", "B2"],  "min": 0, "max": 3000}
FALSE_COLOR = {"bands": ["B12", "B8", "B4"], "min": 0, "max": 3000}
SEV_LEGEND  = {"Low (0.10–0.27)":           "#FFD54F",
               "Moderate-low (0.27–0.44)":  "#FF8F00",
               "Moderate-high (0.44–0.66)": "#D84315",
               "High (> 0.66)":             "#7F0000"}
RESULT_LAYERS = ("Before · true color", "After · false color", "Burn severity (dNBR)", "Study area")

# ---------- Map ----------
m = geemap.Map(height="600px")
outline = {"shapeOptions": {"color": "#FFFFFF", "weight": 2, "fillOpacity": 0}}
m.draw_control.rectangle = outline
m.draw_control.polygon = outline
base = m.layers[0]
base.url = BASEMAPS[basemap.value]
base.name = "Basemap"
base.attribution = "Imagery © Google"
m.centerObject(area.value, 11)
m.add_legend(title="Burn severity (dNBR)", legend_dict=SEV_LEGEND, position="bottomright")


def set_basemap(name):
    base.url = BASEMAPS[name]


def remove_results():
    for c in list(m.controls):
        if isinstance(c, ipyleaflet.SplitMapControl):
            m.remove_control(c)
    for layer in list(m.layers):
        if layer.name in RESULT_LAYERS:
            m.remove_layer(layer)


def update_map(*args):
    r = result.value
    if r is None:
        return
    remove_results()
    left  = geemap.ee_tile_layer(r["before"], TRUE_COLOR,  "Before · true color")
    right = geemap.ee_tile_layer(r["after"],  FALSE_COLOR, "After · false color")
    m.add_layer(left)
    m.add_layer(right)
    m.add_control(ipyleaflet.SplitMapControl(left_layer=left, right_layer=right))
    m.addLayer(r["severity"], {"min": 1, "max": 4, "palette": SEV_COLORS},
               "Burn severity (dNBR)", show_burnt.value)
    m.addLayer(ee.Image().paint(area.value, 0, 2), {"palette": ["#FFFFFF"]}, "Study area")


def severity_chart(df):
    colors = [SEV_COLORS[list(SEVERITY.values()).index(c)] for c in df.columns]
    fig, ax = plt.subplots(figsize=(5, 3))
    df.plot(kind="barh", stacked=True, color=colors, ax=ax, width=0.7)
    ax.set_xlabel("Burnt area (km²)")
    ax.set_ylabel("")
    ax.legend(title="Severity", frameon=False, fontsize=8, title_fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    plt.close(fig)
    return fig


def edge_km2(mask, geom):
    inner = geom.buffer(-500, 10)
    ring  = geom.difference(inner, 10)
    return (ee.Image.pixelArea().updateMask(mask)
            .reduceRegion(reducer=ee.Reducer.sum(), geometry=ring, scale=20, maxPixels=1e10)
            .get("area"))

def burnt_mask(geom, before_dates, after_dates, thr):
    before = s2_composite(before_dates[0], before_dates[1], geom)
    after  = s2_composite(after_dates[0],  after_dates[1],  geom)
    dnbr   = nbr(before).subtract(nbr(after))
    land   = ee.ImageCollection("ESA/WorldCover/v200").first().clip(geom).neq(80)
    return dnbr.gt(thr).And(land)


def fit_area(geom, before_dates, after_dates, thr, tries=4, step=5000):
    for i in range(tries):
        burnt = burnt_mask(geom, before_dates, after_dates, thr)
        touching = ee.Number(edge_km2(burnt, geom)).getInfo() or 0
        if touching < 1e5:
            return geom, i
        geom = geom.buffer(step,10).bounds(10)
    return geom, tries

def analyze():
    status.value = ""
    try:
        result.value = burnscope(area.value,
                                 (before_start.value, before_end.value),
                                 (after_start.value, after_end.value),
                                 threshold.value)
        update_map()
        m.centerObject(area.value)

        touching = (ee.Number(edge_km2(result.value["burnt"], area.value)).getInfo() or 0) / 1e6
        if touching > 0.1:
            status.value = (f"⚠️ The burnt area touches the edge of your box ({touching:.1f} km² along the edge). "
                            "The fire probably continues outside, so draw a bigger box for the full area.")
    except Exception as e:
        status.value = f"⚠️ {e}"


def run():
    try:
        if m.user_roi is not None:
            area.value = m.user_roi
            m.draw_control.clear()
        analyze()
    except Exception as e:
        status.value = f"⚠️ {e}"


def load_event(*args):
    try:
        ev = FIRE_EVENTS[event.value]
        before_start.value, before_end.value = ev["before"]
        after_start.value,  after_end.value  = ev["after"]
        status.value = "🔍 Fitting the box to the fire…"

        geom = ee.Geometry.Rectangle(ev["bbox"]).buffer(3000,10).bounds(10)
        geom, grown = fit_area(geom, ev["before"], ev["after"], threshold.value)
        area.value = geom
        m.draw_control.clear()
        analyze()

        if grown and not status.value:
            status.value = f"🔍 The box was enlarged {grown} time(s) (+{grown * 5} km) to fit the whole fire."
    except Exception as e:
        status.value = f"⚠️ {e}"


def clear_area(*args):
    remove_results()
    m.draw_control.clear()
    result.value = None


# ---------- Page ----------
@solara.component
def Page():
    with solara.Columns([1, 3]):
        with solara.Column():
            solara.Markdown("# 🔥 BurnScope")
            solara.ToggleButtonsSingle(value=mode, values=["Map your own", "Major events"])

            if mode.value == "Major events":
                solara.Select("Event", value=event, values=list(FIRE_EVENTS))
                ev = FIRE_EVENTS[event.value]
                solara.Markdown(f"📅 Before **{ev['before'][0]} → {ev['before'][1]}** · "
                                f"After **{ev['after'][0]} → {ev['after'][1]}**")
                solara.Markdown(ev["info"])
                solara.Markdown(f"[Source]({ev['source']})")
                solara.SliderFloat("Burnt if dNBR above", value=threshold,
                                   min=0.1, max=0.66, step=0.01, thumb_label="always")
                solara.Button("Map this event", on_click=load_event, color="primary")
            else:
                solara.Markdown("**When · before**")
                solara.InputText("From", value=before_start)
                solara.InputText("To", value=before_end)
                solara.Markdown("**When · after**")
                solara.InputText("From", value=after_start)
                solara.InputText("To", value=after_end)
                solara.SliderFloat("Burnt if dNBR above", value=threshold,
                                   min=0.1, max=0.66, step=0.01, thumb_label="always")
                solara.Button("Map burnt area", on_click=run, color="primary")
                solara.Button("New area", on_click=clear_area, outlined=True)

            solara.Markdown(status.value)
            solara.Markdown("**Map layers**")
            solara.Select("Basemap", value=basemap, values=list(BASEMAPS), on_value=set_basemap)
            solara.Checkbox(label="Show burn severity (dNBR)", value=show_burnt, on_value=update_map)

            r = result.value
            if r is not None:
                solara.Markdown(f"### Burnt area: {r['km2']:.1f} km²")
                solara.display(r["table"])
                solara.Markdown("**Burnt area by land cover and severity**")
                solara.FigureMatplotlib(severity_chart(r["sev_df"]))
        with solara.Column():
            solara.display(m)
Page()

Html(layout=None, style_='display: none', tag='span')

Cannot show ipywidgets in text

In [17]:
solara.Button("New area", on_click=clear_area, outlined=True)

Html(layout=None, style_='display: none', tag='span')

Cannot show ipywidgets in text

In [26]:
m

Map(center=[38.010102192506025, 22.599999999999998], controls=(WidgetControl(options=['position', 'transparent…